![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Build the Core

*Project 3 — AI Product: From Model to Product*

---

Build your solution, compare the design choices with evidence, and choose your final version.

> **Done when:** you have an end-to-end result that beats (or honestly fails to beat) the baseline, and you can say why you chose your final version.

---
## 0. Setup

In [ ]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

---
## 1. Model track — compare at least three approaches

On the same folds, or the same validation set, against the baseline from notebook 01. For example:

1. TF-IDF with a linear model (your baseline), tuned.
2. Sentence embeddings (for example `all-MiniLM-L6-v2`) with a classifier.
3. A fine-tuned transformer (DistilBERT on a sample, as in Week 7) or a zero-shot model.

Report the mean and the spread, or the validation score, and the **training time** of each. Track the runs with MLflow if you can.

In [1]:
# %pip install transformers sentencepiece

  Obtaining dependency information for sentencepiece from https://files.pythonhosted.org/packages/c1/ed/8c25ed68a0e9380813f880b0bc3d986f232191e0989d035253c8deb8cfbf/sentencepiece-0.2.3-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
    --------------------------------------- 0.0/1.7 MB 1.3 MB/s eta 0:00:02
   ------ --------------------------------- 0.3/1.7 MB 4.1 MB/s eta 0:00:01
   ------------- -------------------------- 0.6/1.7 MB 4.9 MB/s eta 0:00:01
   -------------------- ------------------- 0.8/1.7 MB 5.9 MB/s eta 0:00:01
   ------------------------------- -------- 1.3/1.7 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------  1.6/1.7 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 6.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline
import time

start = time.perf_counter()

zero_shot = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=-1  # Use CPU
)

loading_time = time.perf_counter() - start

print(f"Model loading time: {loading_time:.2f} seconds")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

c:\Users\rkham\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rkham\.cache\huggingface\hub\models--MoritzLaurer--mDeBERTa-v3-base-mnli-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\rkham\AppData\Local\Programs\Python\Python311\Lib\site-packages\t

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Model loading time: 51.19 seconds


In [3]:
candidate = """
Junior machine learning engineer with Python,
Pandas, scikit-learn, SQL and predictive modeling experience.
"""

job = """
Junior Machine Learning Engineer.
Develop machine learning models using Python,
SQL and scikit-learn.
"""

# Describe both the candidate and the vacancy
text = f"""
Candidate experience:
{candidate}

Job requirements:
{job}
"""

labels = [
    "a strong match between candidate and job",
    "a partial match between candidate and job",
    "a poor match between candidate and job"
]

start = time.perf_counter()

result = zero_shot(
    text,
    candidate_labels=labels,
    hypothesis_template="This candidate-job comparison represents {}.",
    multi_label=False
)

inference_time = time.perf_counter() - start

for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.3f}")

print(f"\nInference time: {inference_time:.2f} seconds")

a partial match between candidate and job: 0.677
a strong match between candidate and job: 0.249
a poor match between candidate and job: 0.074

Inference time: 6.29 seconds


In [4]:
def zero_shot_match(candidate_text, job_text):
    """
    Evaluate how relevant a job is for a candidate.

    Returns a relevance score between 0 and 1.
    """

    text = f"""
    Candidate experience:
    {candidate_text}

    Job description:
    {job_text}
    """

    labels = [
        "a strong match between candidate and job",
        "a partial match between candidate and job",
        "a poor match between candidate and job"
    ]

    result = zero_shot(
        text,
        candidate_labels=labels,
        hypothesis_template="This candidate-job comparison represents {}.",
        multi_label=False,
        truncation=True
    )

    # Convert results into a dictionary
    scores = dict(zip(result["labels"], result["scores"]))

    # Weighted relevance score
    relevance_score = (
        scores[labels[0]] * 1.0
        + scores[labels[1]] * 0.5
        + scores[labels[2]] * 0.0
    )

    return relevance_score

In [5]:
import pandas as pd
import json
import time

jobs = pd.read_csv(
    "../data/evaluation_jobs.csv",
    dtype={"id": str}
)

with open("../data/candidate_profiles.json", encoding="utf-8") as f:
    candidate_profiles = json.load(f)

candidate = candidate_profiles["ml_engineer"]

results = []

start = time.perf_counter()

for _, job in jobs.head(5).iterrows():

    job_text = f"{job['title']}. {job['description']}"

    score = zero_shot_match(candidate, job_text)

    results.append({
        "title": job["title"],
        "zero_shot_score": score
    })

elapsed = time.perf_counter() - start

test_results = pd.DataFrame(results)

display(
    test_results.sort_values(
        "zero_shot_score",
        ascending=False
    )
)

print(f"Total inference time: {elapsed:.2f} seconds")

,title,zero_shot_score
1,Data Scientist (m/w/d),0.672074
4,Computer Vision Intern – Machine Learning Oper...,0.626463
3,AI Engineer (m/w/d) - Implementation & Consulting,0.621427
0,Cloud & AI Engineer Python (m/w/d) 100.000€,0.613409
2,(Junior) Data Scientist (m/w/d) - Datenbankent...,0.510513


Total inference time: 83.48 seconds


In [6]:
import pandas as pd
import time
from pathlib import Path

output = Path("../data/zero_shot_scores.csv")

# Load existing progress if available
if output.exists():
    results_df = pd.read_csv(
        output,
        dtype={"job_id": str}
    )
    results = results_df.to_dict("records")
else:
    results = []

# Track already processed pairs
completed = {
    (row["profile_id"], str(row["job_id"]))
    for row in results
}

start = time.perf_counter()

for profile_id, candidate_text in candidate_profiles.items():

    for _, job in jobs.iterrows():

        job_id = str(job["id"])
        pair = (profile_id, job_id)

        # Skip previously processed pairs
        if pair in completed:
            continue

        job_text = f"{job['title']}. {job['description']}"

        score = zero_shot_match(
            candidate_text,
            job_text
        )

        results.append({
            "profile_id": profile_id,
            "job_id": job_id,
            "zero_shot_score": score
        })

        completed.add(pair)

        # Save progress after every prediction
        pd.DataFrame(results).to_csv(
            output,
            index=False
        )

    print(f"Completed: {profile_id}")

elapsed = time.perf_counter() - start

print(f"\nTotal predictions saved: {len(results)}")
print(f"Execution time this run: {elapsed:.2f} seconds")

Completed: ml_engineer
Completed: backend_developer
Completed: data_analyst
Completed: cloud_engineer
Completed: technical_support

Total predictions saved: 140
Execution time this run: 2156.71 seconds


---
## 2. Model track — tune and choose

Tune the best one or two on the training data only. Choose your final approach, and say why: the score, but also the training time, the size, and how hard it is to run in the app.

---
## 3. LLM track — the pipeline

Build it end to end, with the functions in `src/functions.py`:

- **Retrieval:** embed the chunks, store them (Chroma), retrieve the top `k` with metadata filters.
- **Generation:** a grounded prompt that answers only from the retrieved texts, cites their ids, and says "I don't know" otherwise.
- **Or an agent:** tools with clear descriptions (a retrieval tool, a calculator, web search), a step limit, and a trace you can read.

Run it on three or four questions from your evaluation set and read the answers in full.

---
## 4. LLM track — one design choice, compared

Chunk size, `k`, a metadata filter, the embedding model, the prompt or the tools: change **one** thing and measure it on the evaluation set (at least the retrieval hit rate). Keep the better version, and log the cost of each run.

---
## 5. The final version

Which version you will evaluate in notebook 03 and use in the app, and why. Save it: the fitted pipeline with `joblib` (model track), or the index and the configuration (LLM track), in `models/`.